This notebook takes pre-processed pickle file and DAQ dataframe and matches them. <br> If no DAQ data exists, then re-saves the non-sync data as all_data.

## User Input
Choose which dataset to work on

In [ ]:
%run -i ../src/General_Data/Analysis/Data_Classes.py
pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = January_2024_571.return_params()
pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = January_2024_241.return_params()

# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = March_2024_571.return_params()
# # pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = March_2024_241.return_params()

# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = April_2024_571.return_params()
# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = April_2024_241.return_params()

pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = June_2024_571.return_params()

# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = October_2024_571.return_params()

# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = January_2025_241.return_params()
# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = January_2025_571.return_params()

# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = test_571.return_params()
# pathlist,screen,save_loc,empty,prefixes,DAQ_Matching,bg_file,raw_vcc = test_241.return_params()


## Import statements

In [2]:
import pandas as pd
import yaml
import copy
import numpy as np
import os

## Manipulate Inputs

In [3]:
screen_nickname = screen.split(':')[2]
os.makedirs(save_loc, exist_ok=True)

## Load Data


In [4]:
non_sync_data = pd.read_pickle(save_loc + 'non_sync_data_' + screen_nickname + '.pkl')

if DAQ_Matching is not None:
    with open(DAQ_Matching, 'r') as file:
        matching = yaml.safe_load(file)
    matching = pd.DataFrame(matching['data'])
    DAQ_Data = pd.read_pickle(save_loc + "All_DAQ_Data.pkl")
    idx = [i for i, s in enumerate(DAQ_Data.columns) if screen_nickname+'_images' in s]
    DAQ_Data.rename(columns={list(DAQ_Data.columns[idx])[0]:screen},inplace=True)
    idx = [i for i, s in enumerate(DAQ_Data.columns) if screen_nickname+'_nrows' in s]
    DAQ_Data.rename(columns={list(DAQ_Data.columns[idx])[0]:screen.split('Image:')[0] + 'Image:ArraySize1_RBV'},inplace=True)
    idx = [i for i, s in enumerate(DAQ_Data.columns) if screen_nickname+'_ncols' in s]
    DAQ_Data.rename(columns={list(DAQ_Data.columns[idx])[0]:screen.split('Image:')[0] + 'Image:ArraySize0_RBV'},inplace=True)

## Match DAQ

In [5]:
if DAQ_Matching is not None:
    # Get common columns between DAQ 
    common_idxs = np.array(matching.columns)[np.in1d(np.array(matching.columns),np.array(non_sync_data.columns))]
    # common_idxs = np.delete(common_idxs,7)


    # Remove scan indices
    scan_idxs = common_idxs[np.in1d(common_idxs,np.array(DAQ_Data.columns))]
    common_idxs = common_idxs[~np.in1d(common_idxs,scan_idxs)]
    common_idxs

    ## Explicitly match non-DAQ data to DAQ scans
    # Gets all the other quad settings for the DAQ scans and associates them with a DAQ number
    row_list = []
    for i, row in matching.iterrows():
        non_sync_data_test = copy.deepcopy(non_sync_data)
        for idx in list(common_idxs):
            non_sync_data_test = non_sync_data_test[np.abs(non_sync_data_test[idx]-row[idx])<0.001]
        if len(non_sync_data_test)==1:
            non_sync_data_test.head()

            df_test=pd.Series(non_sync_data_test.iloc[0])
            row_new = df_test.to_dict()
            row_new.update(row)
            row_list.append(row_new)

    new_df = pd.DataFrame(row_list)

    ## Give a clear index to match
    new_df['DAQ_int'] = new_df['DAQ_num'].astype(int)
    DAQ_Data['DAQ_int'] = DAQ_Data['DAQ_num'].astype(int)

    ## Keep only relevant columns in dataframe matched (settings, not readbacks) & relevant rows
    cols_to_keep = [i for i in new_df.columns if 'BCTRL' in i or 'DES' in i or 'DAQ' in i]
    matching_data = new_df[cols_to_keep]
    matching_data = matching_data[matching_data['DAQ_int']>0]

    ## Get all DAQ data matched with experimental data
    combined_data = DAQ_Data.merge(matching_data,how='inner', on='DAQ_int')

    # combined_data[screen]

## Merge DAQ Data

In [6]:
if DAQ_Matching is not None:
    all_data = pd.concat([non_sync_data,combined_data])
else:
    all_data = non_sync_data
# all_data[screen]

## Save Data

In [7]:
all_data.to_pickle(save_loc + "All_Data_" + screen_nickname + '.pkl')